# Session 1: who is in a payment, and what do they say to each other?
**A. Real-world problem.** You scan a code at a shop, type a PIN, and a second later both
phones say "success". No coins moved. What actually happened, and who talked to whom?

**Plan.** (1) meet the cast, (2) write the messages as a list, (3) follow the numbers in
three ledgers, (4) check that no money appears or vanishes on the way, (5) draw the
sequence diagram.

General education about how such systems are organised; not advice about any real account or app.

In [ ]:
import pathlib

import numpy as np

OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

## Step 1: the cast
* **Payer app** and **payee app**: windows on your phones. They hold no money. They only send
  and show messages.
* **Payer bank** and **payee bank**: keep the *ledgers*, lists of numbers saying how much each
  customer is owed by the bank.
* **Switch** (a shared network in the middle): passes messages between apps and banks and keeps
  track of a payment while it is on its way. In this repo it is a toy, not any real network.

The key idea: **money moves as changes of numbers in ledgers, not as coins.**

In [ ]:
ACTORS = ["Payer app", "Switch", "Payer bank", "Payee bank", "Payee app"]

# (from, to, what is said). Message k is step k of the flow.
MESSAGES = [
    ("Payer app", "Switch", "pay Rs X to payee (id T1)"),
    ("Switch", "Payer bank", "can you debit Rs X?"),
    ("Payer bank", "Switch", "debited (payer ledger -X)"),
    ("Switch", "Payee bank", "credit Rs X"),
    ("Payee bank", "Switch", "credited (payee ledger +X)"),
    ("Switch", "Payee app", "success (and to payer app)"),
]

## Step 2: follow the numbers
Between step 3 and step 5 the money has left the payer's ledger but has not yet reached the
payee's ledger. We count it in an **in-flight** box at the switch, so every rupee always has
exactly one home. `flow_states` returns the three numbers after each step.

In [ ]:
def flow_states(payer, payee, amount):
    """Balances after step 0 (before), 1, ... 6. Each is a dict payer / in_flight / payee (rupees).

    If amount is more than the payer has, the payer bank refuses at step 3 and nothing moves."""
    if amount <= 0:
        raise ValueError("amount must be positive")
    declined = amount > payer
    states = []
    for step in range(7):
        s = {"payer": payer, "in_flight": 0, "payee": payee}
        if not declined and step >= 3:
            s["payer"], s["in_flight"] = payer - amount, amount
        if not declined and step >= 5:
            s["in_flight"], s["payee"] = 0, payee + amount
        states.append(s)
    return states


def total(state):
    return state["payer"] + state["in_flight"] + state["payee"]


def conserved(states):
    """True when the total is the same at every step."""
    return len({total(s) for s in states}) == 1

## Step 3: draw the sequence diagram
Time runs downwards. Each arrow is one message. Under each arrow we print the three numbers.

In [ ]:
def draw_sequence(states, amount, path):
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(10, 6.2))
    xs = {a: i for i, a in enumerate(ACTORS)}
    for a, x in xs.items():
        ax.plot([x, x], [0, -6.6], color="#9aa5b1", lw=1, zorder=0)
        ax.text(x, 0.25, a, ha="center", va="bottom", fontsize=9, weight="bold")
    for k, (src, dst, text) in enumerate(MESSAGES, start=1):
        y = -k
        ax.annotate("", xy=(xs[dst], y), xytext=(xs[src], y), arrowprops=dict(arrowstyle="->", color="#2a6fdb", lw=1.5))
        ax.text((xs[src] + xs[dst]) / 2, y + 0.1, f"{k}. {text.replace('X', str(amount))}", ha="center", va="bottom", fontsize=8)
        s = states[k]
        ax.text(4.55, y, f"payer {s['payer']}  in flight {s['in_flight']}  payee {s['payee']}", fontsize=7, va="center", color="#444")
    ax.text(4.55, 0.25, "ledger numbers (rupees)", fontsize=8, weight="bold", va="bottom")
    ax.set_xlim(-0.6, 6.6)
    ax.set_ylim(-6.8, 0.9)
    ax.axis("off")
    ax.set_title("A payment as messages (time runs downwards)")
    fig.tight_layout()
    fig.savefig(path, dpi=120)

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    states = flow_states(1000, 400, 250)
    print("step  payer  in_flight  payee  total")
    for k, s in enumerate(states):
        print(f"{k:4d} {s['payer']:6d} {s['in_flight']:10d} {s['payee']:6d} {total(s):6d}")
    print("money conserved at every step:", conserved(states))
    dec = flow_states(1000, 400, 1200)
    print("declined (1200 > 1000): final", dec[-1], "conserved:", conserved(dec))
    draw_sequence(states, 250, OUT / "sequence_diagram.png")

    fig, ax = plt.subplots(figsize=(6.5, 4))
    steps = np.arange(7)
    p = [s["payer"] for s in states]
    f = [s["in_flight"] for s in states]
    q = [s["payee"] for s in states]
    ax.bar(steps, p, label="payer ledger", color="#2a6fdb")
    ax.bar(steps, f, bottom=p, label="in flight at switch", color="#e9a23b")
    ax.bar(steps, q, bottom=np.array(p) + np.array(f), label="payee ledger", color="#2a9d8f")
    ax.set_xlabel("step in the flow (0 = before)")
    ax.set_ylabel("rupees")
    ax.set_title("Money moves between ledgers; the total stays 1400")
    ax.legend(fontsize=8, loc="lower right")
    ax.grid(alpha=0.3, axis="y")
    fig.tight_layout()
    fig.savefig(OUT / "ledgers_by_step.png", dpi=120)

## Where this stops
We never asked whether the payer really approved the payment, or what to do if a message is
lost. Session 2 adds the ledger rules; Session 3 adds failures and fraud checks.

## Coding tasks
1. **Easy.** Run `python start.py run session-01/lesson.py` and change the payment to Rs 400. Predict the three numbers after step 3 before you look.
2. **Stretch.** Add a seventh message to `MESSAGES` in which the payee app sends a receipt request, and redraw the sequence diagram. Does any ledger number change?
3. **Explain.** Explain, in your own words, why the total stays constant even though the payer's number goes down.

## Assessment
- **A. Conceptual.** Why do the payer app and the payee app hold no money, and why do we need the in-flight box at the switch?
- **B. Numerical.** The payer has Rs 2000 and the payee has Rs 150. The payer pays Rs 600. Give the three numbers after step 3 and after step 6, and the total each time.
- **C. Graph and simulation interpretation.** In the stacked-bar figure, at which steps is there an orange in-flight block, and what happens to the total height?
- **D. Design and optimisation challenge.** Design: the switch crashes after step 3 and never sends step 4. The payer has lost Rs 250 and the payee has gained nothing. Describe two sensible ways the system could recover, and what each would have to remember.

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    st = L.flow_states(1000, 400, 250)
    checks.append(("there are 7 states: before, then after each of 6 messages", len(st) == 7 and len(L.MESSAGES) == 6))
    checks.append(("worked example: paying 250 ends with payer 750, payee 650, nothing in flight",
                   st[-1] == {"payer": 750, "in_flight": 0, "payee": 650}))
    checks.append(("hand check: after step 3 the 250 is in flight (payer 750, in flight 250, payee 400)",
                   st[3] == {"payer": 750, "in_flight": 250, "payee": 400}))
    checks.append(("before step 3 nothing has moved", st[0] == st[1] == st[2] == {"payer": 1000, "in_flight": 0, "payee": 400}))
    checks.append(("total money is 1400 at every step", all(L.total(s) == 1400 for s in st)))
    checks.append(("conserved() agrees", L.conserved(st)))
    dec = L.flow_states(1000, 400, 1200)
    checks.append(("a payment above the balance is declined: nothing moves, total kept", dec[-1] == dec[0] and L.conserved(dec)))
    exact = L.flow_states(1000, 400, 1000)
    checks.append(("paying exactly the balance works (boundary)", exact[-1]["payer"] == 0 and exact[-1]["payee"] == 1400))
    one_over = L.flow_states(1000, 400, 1001)
    checks.append(("one rupee over the balance is declined", one_over[-1]["payer"] == 1000))
    try:
        L.flow_states(1000, 400, 0)
        zero_rejected = False
    except ValueError:
        zero_rejected = True
    checks.append(("an amount of zero is rejected", zero_rejected))
    checks.append(("every message is between two actors from the cast", all(a in L.ACTORS and b in L.ACTORS for a, b, _ in L.MESSAGES)))
    checks.append(("the apps never hold money: only three number boxes exist", set(st[0]) == {"payer", "in_flight", "payee"}))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
